# CBIS-DDSM: Lesion Segmentation **v2** (pretrained encoder + Focal-Tversky + aug)

Improves the from-scratch Attention U-Net (Dice 0.28) with the three highest-yield levers:
1. **Pretrained MobileNetV2 encoder** (ImageNet) with **attention-gated** skip connections.
2. **Focal-Tversky loss** (alpha=0.7) - built for small-lesion class imbalance (lesions ~1.3% of pixels).
3. **Flip + rotate augmentation** (train only) - cuts the bimodal misses.

Reuses the decode-once memmap cache on `D:\cbis-ddsm\_seg_cache` (fast start), patient-disjoint
split, RTX-pinned. Target: push Dice from ~0.28 toward ~0.5-0.7.


In [ ]:
# ---- GPU: pin to RTX ----
import os
os.environ.setdefault('DML_VISIBLE_DEVICES', '0')
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPUs:', gpus)
if gpus:
    try:
        tf.config.set_visible_devices([gpus[0]], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except Exception as e:
        print('gpu cfg:', e)
print('Visible:', tf.config.get_visible_devices('GPU'))

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2

In [ ]:
# ============================================================ CONFIG
DATA_ROOT  = 'D:/cbis-ddsm'; JPEG_ROOT = DATA_ROOT + '/jpeg'; CSV_DIR = DATA_ROOT + '/csv'
DICOM_INFO = CSV_DIR + '/dicom_info.csv'; SAVE_DIR = 'G:/My Drive/VS_CC/models'
CACHE = 'D:/cbis-ddsm/_seg_cache'
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
TRAIN_CSVS = ['mass_case_description_train_set.csv', 'calc_case_description_train_set.csv']
TEST_CSVS  = ['mass_case_description_test_set.csv',  'calc_case_description_test_set.csv']
SZ = 256; BATCH = 4; EPOCHS = 60; RANDOM_STATE = 2021; MASK_THRESH = 127
IMG_SIZE = SZ
ENCODER_TRAINABLE = False        # frozen ImageNet encoder + trained attention decoder
TV_ALPHA, TV_BETA, TV_GAMMA = 0.7, 0.3, 0.75

## 1. Resolve full image + ROI mask (dicom_info.csv)

In [ ]:
def _to_local(p):
    p = str(p).replace('\\', '/'); i = p.find('jpeg/')
    return os.path.join(JPEG_ROOT, p[i + 5:]) if i >= 0 else p
def build_uid_maps(dinfo):
    di = pd.read_csv(dinfo); di['SeriesDescription'] = di['SeriesDescription'].fillna('')
    di['local'] = di['image_path'].apply(_to_local); di['uid'] = di['SeriesInstanceUID'].astype(str)
    full, mask, allf = {}, {}, {}
    for _, r in di.iterrows():
        allf.setdefault(r['uid'], []).append(r['local']); d = r['SeriesDescription'].lower()
        if 'full' in d: full.setdefault(r['uid'], r['local'])
        elif 'roi mask' in d: mask.setdefault(r['uid'], r['local'])
    return full, mask, allf
FULL_MAP, MASK_MAP, ANY_MAP = build_uid_maps(DICOM_INFO)
def series_uid(p):
    q = str(p).replace('\\', '/').strip().split('/'); return q[-2] if len(q) >= 2 else None
def is_mask_like(g):
    return g is not None and np.mean((g <= 10) | (g >= 245)) > 0.90 and len(np.unique(g)) <= 64
def resolve_mask(p):
    u = series_uid(p)
    if u in MASK_MAP: return MASK_MAP[u]
    for c in ANY_MAP.get(u, []):
        if is_mask_like(cv2.imread(c, cv2.IMREAD_GRAYSCALE)): return c
    return None
def load_tables(names):
    df = pd.concat([pd.read_csv(os.path.join(CSV_DIR, n)) for n in names if os.path.exists(os.path.join(CSV_DIR, n))], ignore_index=True)
    df.columns = [c.strip().replace(' ', '_').lower() for c in df.columns]; return df
def prep(df):
    df['full_path'] = df['image_file_path'].apply(lambda p: FULL_MAP.get(series_uid(p)))
    df['mask_path'] = df['roi_mask_file_path'].apply(resolve_mask)
    df['patient_id'] = df['patient_id'].astype(str); df['image_key'] = df['image_file_path'].astype(str)
    return df[['patient_id', 'image_key', 'full_path', 'mask_path']].dropna(subset=['full_path'])
train_df = prep(load_tables(TRAIN_CSVS)); test_df = prep(load_tables(TEST_CSVS))
ov = set(train_df.patient_id) & set(test_df.patient_id)
if ov: train_df = train_df[~train_df.patient_id.isin(ov)].reset_index(drop=True)
print('Train', len(train_df), 'Test', len(test_df), 'disjoint', set(train_df.patient_id).isdisjoint(set(test_df.patient_id)))

## 2. Memmap cache (reused) + augmenting generator

In [ ]:
_clahe = cv2.createCLAHE(2.0, (8, 8))
def load_img(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    return np.stack([_clahe.apply(cv2.resize(img, (SZ, SZ)))] * 3, -1)
def load_mask(path):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE) if path else None
    if not is_mask_like(m): return None
    return (cv2.resize(m, (SZ, SZ), interpolation=cv2.INTER_NEAREST) >= MASK_THRESH).astype(np.uint8)
def union_mask(mps):
    mk = np.zeros((SZ, SZ), np.uint8)
    for mp in mps:
        u = load_mask(mp)
        if u is not None: mk |= u
    return mk
def make_items(df):
    d = df[df['mask_path'].notna()]; items, pats = [], []
    for key, rows in d.groupby('image_key'):
        items.append((rows.iloc[0]['full_path'], list(rows['mask_path']))); pats.append(rows.iloc[0]['patient_id'])
    return items, np.array(pats)
items_tr, pats_tr = make_items(train_df); items_te, pats_te = make_items(test_df)
print('masked -> train', len(items_tr), 'test', len(items_te))

def _p(prefix, ch): return os.path.join(CACHE, f'{prefix}_{ch}_{SZ}.npy')
def build_cache(items, prefix):
    if os.path.exists(_p(prefix, 'X')) and os.path.exists(_p(prefix, 'Y')):
        print(f'  cache {prefix} exists -> skip'); return
    n = len(items)
    X = np.lib.format.open_memmap(_p(prefix, 'X'), mode='w+', dtype=np.uint8, shape=(n, SZ, SZ, 3))
    Y = np.lib.format.open_memmap(_p(prefix, 'Y'), mode='w+', dtype=np.uint8, shape=(n, SZ, SZ, 1))
    for i, (fp, mps) in enumerate(items):
        img = load_img(fp)
        if img is not None: X[i] = img
        Y[i] = union_mask(mps)[..., None]
        if (i + 1) % 400 == 0: print('   decoded', i + 1)
    X.flush(); Y.flush(); del X, Y
build_cache(items_tr, 'tr'); build_cache(items_te, 'te')
Xtr = np.load(_p('tr', 'X'), mmap_mode='r'); Ytr = np.load(_p('tr', 'Y'), mmap_mode='r')
Xte = np.load(_p('te', 'X'), mmap_mode='r'); Yte = np.load(_p('te', 'Y'), mmap_mode='r')
print('memmaps -> Xtr', Xtr.shape, 'Xte', Xte.shape)

class SegSeq(tf.keras.utils.Sequence):
    def __init__(self, X, Y, idx, batch, shuffle=True, augment=False):
        self.X = X; self.Y = Y; self.idx = np.asarray(idx); self.batch = batch
        self.shuffle = shuffle; self.augment = augment; self.on_epoch_end()
    def __len__(self): return int(np.ceil(len(self.idx) / self.batch))
    def on_epoch_end(self):
        self.order = np.arange(len(self.idx))
        if self.shuffle: np.random.shuffle(self.order)
    def __getitem__(self, b):
        sel = np.sort(self.idx[self.order[b*self.batch:(b+1)*self.batch]])
        X = self.X[sel].astype('float32'); Y = self.Y[sel].astype('float32')
        if self.augment:
            for k in range(len(X)):
                if np.random.rand() < 0.5: X[k] = X[k, :, ::-1]; Y[k] = Y[k, :, ::-1]
                if np.random.rand() < 0.5: X[k] = X[k, ::-1]; Y[k] = Y[k, ::-1]
                r = np.random.randint(4)
                if r: X[k] = np.rot90(X[k], r); Y[k] = np.rot90(Y[k], r)
        return X, Y

## 3. MobileNetV2-encoder Attention U-Net

In [ ]:
def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    return x
def attn_gate(skip, gating, inter):
    tx = layers.Conv2D(inter, 1)(skip); pg = layers.Conv2D(inter, 1)(gating)
    r = layers.Activation('relu')(layers.Add()([tx, pg]))
    psi = layers.Conv2D(1, 1, activation='sigmoid')(r)
    return layers.Multiply()([skip, psi])

def build_model(sz=SZ):
    base = MobileNetV2(input_shape=(sz, sz, 3), include_top=False, weights='imagenet')
    base.trainable = ENCODER_TRAINABLE
    names = ['block_1_expand_relu', 'block_3_expand_relu', 'block_6_expand_relu',
             'block_13_expand_relu', 'block_16_project']         # 128,64,32,16,8
    encoder = Model(base.input, [base.get_layer(n).output for n in names])
    inp = layers.Input((sz, sz, 3)); x = layers.Rescaling(1./127.5, offset=-1)(inp)
    s128, s64, s32, s16, s8 = encoder(x)
    def up(x, skip, f):
        x = layers.UpSampling2D()(x); a = attn_gate(skip, x, f)
        return conv_block(layers.Concatenate()([x, a]), f)
    d = up(s8, s16, 256); d = up(d, s32, 128); d = up(d, s64, 64); d = up(d, s128, 32)
    d = conv_block(layers.UpSampling2D()(d), 16)                 # 128 -> 256
    out = layers.Conv2D(1, 1, activation='sigmoid')(d)
    return Model(inp, out)

## 4. Focal-Tversky loss + metrics

In [ ]:
def _flat(y_true, y_pred):
    return tf.reshape(tf.cast(y_true, tf.float32), [-1]), tf.reshape(y_pred, [-1])
def tversky_index(y_true, y_pred, a=TV_ALPHA, b=TV_BETA, smooth=1.):
    yt, yp = _flat(y_true, y_pred)
    tp = tf.reduce_sum(yt*yp); fn = tf.reduce_sum(yt*(1-yp)); fp = tf.reduce_sum((1-yt)*yp)
    return (tp + smooth) / (tp + a*fn + b*fp + smooth)
def focal_tversky_loss(y_true, y_pred):
    return tf.pow(1 - tversky_index(y_true, y_pred), TV_GAMMA)
def dice_coef(y_true, y_pred, smooth=1.):
    yt, yp = _flat(y_true, y_pred); inter = tf.reduce_sum(yt*yp)
    return (2*inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)
def iou_metric(y_true, y_pred, smooth=1.):
    yt, yp = _flat(y_true, tf.cast(y_pred > 0.5, tf.float32)); inter = tf.reduce_sum(yt*yp)
    return (inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) - inter + smooth)

model = build_model()
model.compile(Adam(1e-3), loss=focal_tversky_loss, metrics=[dice_coef, iou_metric])
print('params %.2fM | encoder_trainable=%s' % (model.count_params()/1e6, ENCODER_TRAINABLE))

## 5. Train (patient-disjoint split; augmented train generator)

In [ ]:
gss = GroupShuffleSplit(1, test_size=0.15, random_state=RANDOM_STATE)
tr, va = next(gss.split(np.arange(len(items_tr)), groups=pats_tr))
train_gen = SegSeq(Xtr, Ytr, tr, BATCH, shuffle=True, augment=True)
val_gen   = SegSeq(Xtr, Ytr, va, BATCH, shuffle=False, augment=False)
print('fit', len(tr), 'val', len(va))
hist = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
                 callbacks=[EarlyStopping(monitor='val_dice_coef', mode='max', patience=12, restore_best_weights=True),
                            ReduceLROnPlateau(monitor='val_dice_coef', mode='max', factor=0.5, patience=5, min_lr=1e-5)],
                 verbose=2)

## 6. Evaluate on the patient-disjoint test set

In [ ]:
P = model.predict(Xte, batch_size=BATCH, verbose=0)[..., 0]
def per_image(y, p, thr=0.5):
    d, io, hit = [], [], []
    for i in range(len(y)):
        gt = np.asarray(y[i, :, :, 0]).astype(bool); pm = p[i] >= thr
        inter = np.logical_and(gt, pm).sum(); s = gt.sum() + pm.sum(); u = np.logical_or(gt, pm).sum()
        d.append(2*inter/s if s else 1.0); io.append(inter/u if u else 1.0)
        yy, xx = np.unravel_index(int(np.argmax(p[i])), p[i].shape); hit.append(int(gt[yy, xx]))
    return np.array(d), np.array(io), np.array(hit)
d, io, hit = per_image(Yte, P)
# also Dice on lesion-present images only (excludes trivial empties)
print(f'TEST  Dice {d.mean():.3f} +/- {d.std():.3f} | IoU {io.mean():.3f} | pointing-game {hit.mean()*100:.1f}%  (n={len(d)})')
print(f'      Dice(>0.1) hit-rate {np.mean(d > 0.1)*100:.1f}% | median Dice {np.median(d):.3f}')

### 6b. Overlays: image + GT (green) + prediction (red)

In [ ]:
order = np.argsort(-d)[:6]
fig, ax = plt.subplots(1, 6, figsize=(18, 3.2), dpi=110)
for a, idx in zip(ax, order):
    a.imshow(np.asarray(Xte[idx])[..., 0], cmap='gray')
    a.contour(np.asarray(Yte[idx, :, :, 0]), colors='lime', linewidths=1.2)
    a.contour((P[idx] >= 0.5).astype(float), colors='red', linewidths=1.0)
    a.set_title(f'Dice {d[idx]:.2f}', fontsize=9); a.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'cbis_unet_v2_overlays.png'), dpi=200, bbox_inches='tight'); plt.show()

## 7. Summary + save

In [ ]:
summary = (
    f"MobileNetV2-encoder Attention U-Net (frozen ImageNet encoder, attention-gated skips, "
    f"Focal-Tversky loss a={TV_ALPHA}, flip/rotate aug) for CBIS-DDSM lesion segmentation "
    f"(patient-disjoint split, {SZ}x{SZ}). Test: Dice {d.mean():.3f} +/- {d.std():.3f}, "
    f"IoU {io.mean():.3f}, pointing-game {hit.mean()*100:.1f}% over {len(d)} masked test "
    f"mammograms. (v1 from-scratch U-Net: Dice 0.282; weakly-supervised attention: ~0-5%.)"
)
print(summary)
open(os.path.join(SAVE_DIR, 'cbis_unet_v2_summary.txt'), 'w').write(summary)
model.save_weights(os.path.join(SAVE_DIR, 'cbis_attention_unet_v2.weights.h5'))
print('\nSaved to', SAVE_DIR)